# 第 24 课 · FP32、FP16、BF16 与自动混合精度

**主线：理解数值格式 → 观察算子精度 → 修复梯度下溢 → 正确训练 → 核对资源口径。**

你已经实现了 RoPE、GQA 与 SDPA。本课保持这些基本计算含义，不再扩展注意力结构，而是研究：同一组数学公式用不同精度执行时，什么会变化、什么不应该变化。

完成后，你应能解释“BF16 范围大但不一定更精细”，写出正确的 AMP 训练循环，并区分参数存储精度、中间结果精度、梯度精度和内部累积精度。

本讲义**完全独立**，不导入同名脚本或之前的课程；配套 `.py` 使用相同的核心实现。CPU 可以完成全部数值原理，以及实际支持的 BF16 训练。CUDA/FP16 路径留给你在本机验证，不伪造 3060 成绩。没有模型/语料下载，没有安装命令自动执行。

建议按两次实验完成：先学到 GradScaler，再做完整解码器对照与练习。实际输出已经保留，但请重新运行并修改例子。

## 0. 运行约定

沿用已经安装的 PyTorch 环境。本次作者运行环境是 `PyTorch 2.10.0+cpu`，不是要求你安装某个“最新版本”。本课使用现代接口 `torch.autocast` 与 `torch.amp.GradScaler`，不使用旧的 `torch.cuda.amp.*`。

Notebook 默认在**当前工作目录**的 `lesson_24_outputs` 写入报告和模型；同名生成文件会覆盖，不修改其他课文件。需要保留旧实验时，改 `OUTPUT_DIR`。使用 **Restart Kernel and Run All** 检查是否依赖旧变量。

先用 `DEVICE_NAME="cpu"`；确认本机 CUDA 可用后改为 `"cuda"`。`"cuda"` 不可用会报错，不会悄悄回退。`MODE="all"` 会探测并运行可用分支。

来源：[PyTorch AMP](https://docs.pytorch.org/docs/2.10/amp.html)。

In [1]:
import argparse
from contextlib import contextmanager, nullcontext
import copy
from dataclasses import asdict, dataclass
import gc
import hashlib
import json
import math
from pathlib import Path
import platform
import tempfile
from typing import Callable

import torch
from torch import nn
from torch.nn import functional as F

SOURCES = {
    "amp": "https://docs.pytorch.org/docs/stable/amp.html",
    "examples": "https://docs.pytorch.org/docs/stable/notes/amp_examples.html",
    "recipe": "https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html",
    "finfo": "https://docs.pytorch.org/docs/stable/type_info.html",
    "formats": "https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/data-format-desc.html",
    "paper": "https://arxiv.org/abs/1710.03740",
    "numerics": "https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html",
    "bf16_probe": "https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_bf16_supported.html",
    "memory": "https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html",
}
MODES = ("fp32", "amp_fp16", "amp_bf16")
MODE_DTYPES = {"fp32": torch.float32, "amp_fp16": torch.float16,
               "amp_bf16": torch.bfloat16}


def positive_int(value, name):
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受 bool")


def validate_mode(mode):
    if mode not in MODES:
        raise ValueError(f"mode 必须是 {MODES} 之一")


def choose_device(name):
    if name not in ("cpu", "cuda", "auto"):
        raise ValueError("device 只能是 cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请使用 --device cpu，或检查本机安装。不会静默回退。")
    return torch.device(name)


def amp_context(mode, device):
    validate_mode(mode)
    if mode == "fp32":
        # 显式关闭，避免被外层 notebook 的 autocast 环境意外影响。
        return torch.autocast(device_type=device.type, enabled=False)
    return torch.autocast(device_type=device.type, dtype=MODE_DTYPES[mode])


@contextmanager
def fp32_reference_policy():
    """局部固定 CUDA FP32 矩阵策略，退出恢复；不声称禁用所有数值近似。

    本课 eager、无卷积。新旧精度 API 不混用。低精度 GEMM 的内部累积仍由后端决定。
    """
    matmul = torch.backends.cuda.matmul
    if hasattr(matmul, "fp32_precision"):
        old = matmul.fp32_precision
        matmul.fp32_precision = "ieee"
        try:
            yield {"api": "cuda.matmul.fp32_precision", "value": str(matmul.fp32_precision)}
        finally:
            matmul.fp32_precision = old
    else:
        old = torch.get_float32_matmul_precision()
        torch.set_float32_matmul_precision("highest")
        try:
            yield {"api": "set_float32_matmul_precision", "value": "highest"}
        finally:
            torch.set_float32_matmul_precision(old)


def environment(device):
    result = {"python": platform.python_version(), "torch": str(torch.__version__),
              "cuda_build": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
              "device": str(device), "threads": torch.get_num_threads(),
              "gpu": None, "gpu_total_bytes": None, "bf16_native_reported": None,
              "bf16_with_emulation_reported": None,
              "matmul_precision": torch.get_float32_matmul_precision(),
              "note": "dtype 属性不暴露全部内部累积精度；CPU 结果不是 GPU 性能代理。"}
    if device.type == "cuda":
        prop = torch.cuda.get_device_properties(device)
        result.update(gpu=prop.name, gpu_total_bytes=prop.total_memory,
                      capability=list(torch.cuda.get_device_capability(device)))
        try:
            result["bf16_native_reported"] = torch.cuda.is_bf16_supported(including_emulation=False)
            result["bf16_with_emulation_reported"] = torch.cuda.is_bf16_supported(including_emulation=True)
        except TypeError:
            result["bf16_with_emulation_reported"] = torch.cuda.is_bf16_supported()
    return result

In [2]:
DEVICE_NAME = "cpu"      # 本机可改为 "cuda" 或 "auto"
MODE = "all"             # all / fp32 / amp_fp16 / amp_bf16
STEPS = 80
BATCH_SIZE = 16
OUTPUT_DIR = Path("lesson_24_outputs")

DEVICE = choose_device(DEVICE_NAME)
# 小模型单线程使 CPU 教学实验更稳定；本 Notebook 显式改变此设置。
torch.set_num_threads(1)
print(json.dumps(environment(DEVICE), ensure_ascii=False, indent=2))
print("实际输出位置：", OUTPUT_DIR.resolve())

{
  "python": "3.13.5",
  "torch": "2.10.0+cpu",
  "cuda_build": null,
  "cuda_available": false,
  "device": "cpu",
  "threads": 1,
  "gpu": null,
  "gpu_total_bytes": null,
  "bf16_native_reported": null,
  "bf16_with_emulation_reported": null,
  "matmul_precision": "highest",
  "note": "dtype 属性不暴露全部内部累积精度；CPU 结果不是 GPU 性能代理。"
}
实际输出位置： /mnt/data/lesson24_validation/notebook_run/lesson_24_outputs


## 1. 同样叫“16 位”，为什么不是同一种精度？

浮点数不是一个固定小数位数的十进制数。对正规数，可以粗略理解为：

\[
(-1)^s\times(1+f)\times 2^e
\]

符号决定正负，指数控制范围，尾数部分控制同一数量级内的细粒度。

| 格式 | 总位数 | 符号位 | 指数位 | 存储的尾数小数位 |
|---|---:|---:|---:|---:|
| FP32 | 32 | 1 | 8 | 23 |
| FP16 | 16 | 1 | 5 | 10 |
| BF16 | 16 | 1 | 8 | 7 |

BF16 与 FP32 有相同的指数位数，但最大有限值并不精确相同。FP16 的范围小得多，却比 BF16 有更多尾数位。**“范围”与“间隔”是两件事。**

来源：[NVIDIA 数据格式](https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/data-format-desc.html)。

In [3]:
def format_table():
    rows = []
    for name, dtype, exponent, fraction in [
        ("FP32", torch.float32, 8, 23), ("FP16", torch.float16, 5, 10),
        ("BF16", torch.bfloat16, 8, 7),
    ]:
        f = torch.finfo(dtype)
        rows.append({"name": name, "bits": f.bits, "bytes": f.bits // 8,
                     "exponent_bits": exponent, "stored_fraction_bits": fraction,
                     "max": f.max, "smallest_normal": f.tiny, "eps_at_1": f.eps,
                     "theoretical_smallest_subnormal": f.tiny * f.eps})
    return rows


def scalar_text(value):
    if math.isnan(value):
        return "NaN"
    if math.isinf(value):
        return "+Infinity" if value > 0 else "-Infinity"
    return value


def representation_demo():
    # CPU 做存储格式参考；不把 subnormal 存储能力等同于每个 GPU 内核的运算行为。
    values = [1.0 + 2.0**-9, 100000.0, 1e-8]
    result = {"source_fp64": values, "casts": {}, "small_updates": {}}
    for name, dtype in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
        cast = torch.tensor(values, dtype=torch.float64).to(dtype).double()
        result["casts"][name] = [scalar_text(float(v)) for v in cast]
        w = torch.tensor(1.0, dtype=dtype)
        for _ in range(100):
            w.sub_(1e-4)
        result["small_updates"][name] = float(w)
    x = torch.tensor([300.0, 400.0], dtype=torch.float16)
    result["square_before_upcast"] = [scalar_text(float(v)) for v in x.square().float()]
    result["square_after_upcast"] = x.float().square().tolist()
    result["casting_back_does_not_recover"] = float(torch.tensor(1.0 + 2.0**-9).bfloat16().float())
    return result

In [4]:
for row in format_table():
    print(f"{row['name']:5} | bytes={row['bytes']} | max={row['max']:.8g} "
          f"| eps={row['eps_at_1']:.8g} | tiny={row['smallest_normal']:.8g}")

FP32  | bytes=4 | max=3.4028235e+38 | eps=1.1920929e-07 | tiny=1.1754944e-38
FP16  | bytes=2 | max=65504 | eps=0.0009765625 | tiny=6.1035156e-05
BF16  | bytes=2 | max=3.3895314e+38 | eps=0.0078125 | tiny=1.1754944e-38


### `eps` 和 `tiny` 分别是什么意思？

`torch.finfo(dtype).eps` 是 **1 与大于 1 的下一个可表示值的间隔**，不是所有数值处都一样的“最小误差”。

`tiny` 是最小正**正规数**，不是最小正数；格式还允许更小的次正规数（subnormal）。不同硬件/算子的次正规数处理可能不同，不能从存储格式直接推断所有计算内核的行为。

先预测：把 `[1.001953125, 100000, 1e-8]` 转成 FP16、BF16，哪个会溢出？哪个会变成 0？哪个会变成 1？

来源：[PyTorch Type Info](https://docs.pytorch.org/docs/stable/type_info.html)。

In [5]:
representation = representation_demo()
print("原数：", representation["source_fp64"])
for dtype, values in representation["casts"].items():
    print(dtype, values)
print("BF16 舍入后转回 FP32：", representation["casting_back_does_not_recover"])

原数： [1.001953125, 100000.0, 1e-08]
fp32 [1.001953125, 100000.0, 9.99999993922529e-09]
fp16 [1.001953125, '+Infinity', 0.0]
bf16 [1.0, 99840.0, 1.0011717677116394e-08]
BF16 舍入后转回 FP32： 1.0


### 实验解释：转回 FP32 不会找回已经丢掉的位

FP16 存储 `100000` 得到无穷大，BF16 存储同一个数保持有限但变成 `99840`。BF16 将 `1.001953125` 舍入为 `1.0`，再转回 FP32 也仍是 `1.0`。

这是**表示误差**。它与标签移位、mask 写错这类**程序错误**不同：后者不能靠换精度或放宽容差修好。

另外，`inf` 不是很大的普通数；`NaN` 也不是“零”。本课 JSON 用字符串明确记录数值实验里故意产生的非有限值，避免写出不标准的 JSON。

## 2. 为什么不能把所有参数 `.half()` 后直接当作 AMP？

先看最小的更新：从 `w=1.0` 开始，每次减 `0.0001`，重复 100 次。数学结果是 `0.99`。

若参数本身一直用低精度存储，每次更新都可能小到不足以改变当前可表示值。**不是只在最后算一次 1−0.01，而是每一步都发生了舍入。**

混合精度训练的一条常见路线是保留 FP32 参数用于更新，在适合的算子中使用低精度计算。本课采用这一条路线，不能泛化为所有训练框架都采用相同的权重管理方式。

来源：[Mixed Precision Training](https://arxiv.org/abs/1710.03740)。

In [6]:
print("连续 100 次小更新后的参数：")
for dtype, w in representation["small_updates"].items():
    print(dtype, w)
print("平方后再转 FP32：", representation["square_before_upcast"])
print("先转 FP32 再平方：", representation["square_after_upcast"])

连续 100 次小更新后的参数：
fp32 0.9899983406066895
fp16 1.0
bf16 1.0
平方后再转 FP32： ['+Infinity', '+Infinity']
先转 FP32 再平方： [90000.0, 160000.0]


### 三种精度不要混为一谈

| 问题 | 本课会怎样观察 |
|---|---|
| 参数存储是什么 dtype？ | `next(model.parameters()).dtype` |
| 某个算子输出是什么 dtype？ | 在前向中记录 Q/K/V、残差、logits |
| 梯度最终存储是什么 dtype？ | `parameter.grad.dtype` |
| 内核内部怎样累积？ | **不能仅凭输出 dtype 判断**，需后端文档或更专门分析 |

在适当硬件与策略下，FP32 矩阵乘也可能使用 TF32 等内部路径；它不是把存储张量变成一个 `torch.tf32` 类型。本课给 CUDA FP32 参考分支局部设置矩阵策略并记录，退出后恢复。没有因此声称控制了所有内部实现细节。

来源：[PyTorch 数值精度说明](https://docs.pytorch.org/docs/2.10/notes/numerical_accuracy.html)。

## 3. 梯度下溢：loss 有限，参数也可能学不到

假设某个 FP16 中间位置的真实梯度是 $10^{-8}$。即使 loss 用 FP32 计算，反向传播经过 FP16 中间结果时，也可能把它舍入成 0。

Loss scaling 在反传**之前**放大 loss：

\[
L_s=S L,\qquad \nabla L_s=S\nabla L
\]

更新参数前再除回 $S$。理想精确运算下梯度不变；数值上，较大的中间梯度更可能被 FP16 表示。

它不是把学习率扩大 $S$ 倍，也不是在梯度已经变成 0 后乘一个大数。

来源：[PyTorch Gradient Scaling](https://docs.pytorch.org/docs/2.10/amp.html#gradient-scaling)。

In [7]:
def underflow_demo(scale=65536.0):
    """loss 是 FP32，但反传穿过 FP16 cast；可在 CPU 观察梯度消失。

    这是有意构造的数值机制，不是实际模型的性能/准确率结论。
    """
    if not math.isfinite(scale) or scale <= 0:
        raise ValueError("scale 必须正且有限")
    def once(s):
        w = torch.tensor(1.0, requires_grad=True)
        loss = w.to(torch.float16).float() * 1e-8
        (loss * s).backward()
        return float(loss.detach()), float(w.grad), float(w.grad / s)
    plain = once(1.0)
    scaled = once(scale)
    forward_bad = (torch.tensor(1.0).half() * 100000.0).float()
    return {"expected_gradient": 1e-8, "loss_fp32": plain[0],
            "unscaled_path_gradient": plain[2], "scaled_leaf_gradient": scaled[1],
            "unscaled_after_backward": scaled[2], "scale": scale,
            "scaling_zero_after_backward": plain[2] * scale,
            "forward_overflow": scalar_text(float(forward_bad)),
            "scaling_overflowed_loss": scalar_text(float(forward_bad * 0.01))}


def clipping_order_demo(scale=128.0):
    positive = math.isfinite(scale) and scale > 0
    if not positive:
        raise ValueError("scale 必须正且有限")
    gradient = torch.tensor([3.0, 4.0])
    correct, wrong = nn.Parameter(torch.zeros(2)), nn.Parameter(torch.zeros(2))
    correct.grad = gradient.clone() * scale
    correct.grad.div_(scale)
    nn.utils.clip_grad_norm_([correct], 1.0)
    wrong.grad = gradient.clone() * scale
    nn.utils.clip_grad_norm_([wrong], 1.0)
    wrong.grad.div_(scale)
    return {"original_norm": 5.0, "scale": scale,
            "unscale_then_clip": correct.grad.tolist(),
            "clip_then_unscale": wrong.grad.tolist(),
            "correct_norm": float(correct.grad.norm()), "wrong_norm": float(wrong.grad.norm())}


def scaler_control_demo(device=torch.device("cpu")):
    """真实 GradScaler；故意注入非有限梯度，观察跳过与降 scale。

    即使在 CPU 上也验证状态控制；这里不使用 autocast，不把它称为 CUDA FP16 训练。
    """
    p = nn.Parameter(torch.tensor(1.0, device=device))
    opt = torch.optim.SGD([p], lr=0.1)
    scaler = torch.amp.GradScaler(device.type, init_scale=128.0, growth_interval=2)
    count = {"steps": 0}
    def after_step(*_):
        count["steps"] += 1
    handle = opt.register_step_post_hook(after_step)
    rows = []
    for inject_bad in (False, True, False):
        opt.zero_grad(set_to_none=True)
        before = float(p.detach())
        old_count = count["steps"]
        old_scale = scaler.get_scale()
        loss = p.square()   # 前向 loss 始终有限
        scaler.scale(loss).backward()
        if inject_bad:
            p.grad.fill_(float("inf"))  # 在 unscale 检查之前注入
        scaler.unscale_(opt)
        if bool(torch.isfinite(p.grad).all()):
            nn.utils.clip_grad_norm_([p], 10.0)
        scaler.step(opt)
        scaler.update()
        rows.append({"injected_inf": inject_bad, "before": before, "after": float(p.detach()),
                     "scale_before": old_scale, "scale_after": scaler.get_scale(),
                     "optimizer_ran": count["steps"] > old_count,
                     "loss_finite": bool(torch.isfinite(loss.detach()))})
    handle.remove()
    return {"device": str(device), "rows": rows, "actual_updates": count["steps"]}

In [8]:
underflow = underflow_demo()
for key, value in underflow.items():
    print(key, "=", value)

expected_gradient = 1e-08
loss_fp32 = 9.99999993922529e-09
unscaled_path_gradient = 0.0
scaled_leaf_gradient = 0.0006551742553710938
unscaled_after_backward = 9.997165761888027e-09
scale = 65536.0
scaling_zero_after_backward = 0.0
forward_overflow = +Infinity
scaling_overflowed_loss = +Infinity


### 读结果，而不是只看“运行成功”

本实验中，不缩放的路径返回梯度 `0`；先缩放、反传后再还原，结果约为 `9.997×10⁻⁹`，接近而不是精确等于 `10⁻⁸`。

这里 `w` 是 FP32 叶子参数，但中间经过 FP16 cast，所以 `.grad.dtype == float32` 也**不代表整个反传从未经历低精度**。

最后两项显示另一个边界：前向结果已经是 `inf` 时，乘一个有限缩放因子仍然是 `inf`。GradScaler 不能挽救已经发生的前向溢出，也不能修复数据错误和除零。

## 4. 梯度裁剪必须发生在去缩放之后

真实梯度是 `[3,4]`，范数为 5；裁剪阈值为 1。正确输出应该约为 `[0.6,0.8]`。

若先放大 128 倍，再按阈值 1 裁剪，最后除以 128，得到的梯度范数只有 `1/128`。你无意中改变了更新尺度。

正确顺序：**scale → backward → unscale → clip → step → update**。

来源：[PyTorch AMP 的梯度裁剪示例](https://docs.pytorch.org/docs/stable/notes/amp_examples.html#gradient-clipping)。

In [9]:
clipping = clipping_order_demo()
print(json.dumps(clipping, ensure_ascii=False, indent=2))

{
  "original_norm": 5.0,
  "scale": 128.0,
  "unscale_then_clip": [
    0.5999999046325684,
    0.7999998331069946
  ],
  "clip_then_unscale": [
    0.004687500186264515,
    0.0062500000931322575
  ],
  "correct_norm": 0.9999998211860657,
  "wrong_norm": 0.0078125
}


## 5. GradScaler 不只做乘除，还管理异常梯度

我们现在调用真实的 `torch.amp.GradScaler`，连续尝试三次更新：第一次正常，第二次故意把梯度改成 `inf`，第三次恢复正常。

观察：第二次优化器是否被调用？参数是否保持不变？scale 是否下降？第三次能否继续？

**这段可以在 CPU 演示缩放器的控制逻辑，不声称在 CPU 测到了 GPU FP16 内核。** BF16 训练分支在本课不启用 scaler。BF16 的范围大使它通常不依赖 FP16 式的梯度缩放，但 BF16 仍然有舍入误差，不能保证任意网络都稳定。

来源：[PyTorch AMP 示例](https://docs.pytorch.org/docs/stable/notes/amp_examples.html)。

In [10]:
scaler_demo = scaler_control_demo(DEVICE)
for row in scaler_demo["rows"]:
    print(row)
print("实际 optimizer.step 次数：", scaler_demo["actual_updates"])

{'injected_inf': False, 'before': 1.0, 'after': 0.800000011920929, 'scale_before': 128.0, 'scale_after': 128.0, 'optimizer_ran': True, 'loss_finite': True}
{'injected_inf': True, 'before': 0.800000011920929, 'after': 0.800000011920929, 'scale_before': 128.0, 'scale_after': 64.0, 'optimizer_ran': False, 'loss_finite': True}
{'injected_inf': False, 'before': 0.800000011920929, 'after': 0.6399999856948853, 'scale_before': 64.0, 'scale_after': 64.0, 'optimizer_ran': True, 'loss_finite': True}
实际 optimizer.step 次数： 2


### 不要根据 `scaler.step()` 的返回值猜有没有更新

许多优化器的 `step()` 成功执行也返回 `None`。本课使用优化器 step hook 或计数子类记录**实际调用次数**，与训练循环的尝试次数分开。

有一个容易忽视的结论：第二次的前向 loss 仍然有限，梯度却不有限。所以只监控 `loss` 并不足以证明整次更新正常。

缩放器发现非有限梯度会跳过更新并调整 scale；scale 不保证永远大于等于 1。遇到频繁跳步时，应继续定位数值问题，而不是把“没有报错”当作稳定训练。

## 6. Autocast：按算子选择精度，不是统一改模型 dtype

本课用一个两层、GQA + RoPE + RMSNorm + SwiGLU 的小解码器接入 AMP。模型参数从头到尾保留 FP32。

前向和 loss 放进 `autocast`，反传放在外面。哪些操作使用哪种类型，由算子规则与实际输入决定，而不是对所有张量强制一次 `.half()`。

自定义 RMSNorm 不会因为名字像内置算子就自动获得精度保护。我们显式关闭这一小段 autocast，先升到 FP32 计算平方与归一化。RoPE 也明确以 FP32 计算角度和旋转，再转回 Q/K 的 dtype。

来源：[PyTorch 2.10 autocast 约定](https://docs.pytorch.org/docs/2.10/amp.html)。

In [11]:
@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int = 16
    d_model: int = 48
    n_heads: int = 4
    n_kv_heads: int = 2
    n_layers: int = 2
    ff_dim: int = 128
    max_len: int = 16
    eps: float = 1e-6

    def __post_init__(self):
        for name in ("vocab_size", "d_model", "n_heads", "n_kv_heads", "n_layers", "ff_dim", "max_len"):
            positive_int(getattr(self, name), name)
        if self.vocab_size < 8 or self.max_len < 9:
            raise ValueError("本课合成任务要求 vocab_size>=8、max_len>=9")
        if self.d_model % self.n_heads or self.n_heads % self.n_kv_heads:
            raise ValueError("D 必须能被 Hq 整除，Hq 必须能被 Hkv 整除")
        if (self.d_model // self.n_heads) % 2:
            raise ValueError("RoPE 每头维度必须为偶数")
        if not math.isfinite(self.eps) or self.eps <= 0:
            raise ValueError("eps 必须为正有限数")


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # 自定义函数不会仅因名字叫 RMSNorm 就自动获得算子精度规则。
        with torch.autocast(device_type=x.device.type, enabled=False):
            work_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
            work = x.to(work_dtype)
            normalized = work * torch.rsqrt(work.square().mean(-1, keepdim=True) + self.eps)
            result = normalized * self.weight.to(work_dtype)
        return result.to(x.dtype)


def apply_rope(x, inv_freq):
    """Q/K [B,H,T,d]，相邻配对。FP32 角度/中间运算，输出回到 Q/K dtype。"""
    with torch.autocast(device_type=x.device.type, enabled=False):
        work_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
        positions = torch.arange(x.shape[-2], device=x.device, dtype=work_dtype)
        angles = positions[:, None] * inv_freq.to(work_dtype)[None, :]
        c, s = angles.cos()[None, None], angles.sin()[None, None]
        work = x.to(work_dtype)
        even, odd = work[..., 0::2], work[..., 1::2]
        rotated = torch.stack((even*c - odd*s, even*s + odd*c), dim=-1).flatten(-2)
    return rotated.to(x.dtype)

### 接上之前熟悉的结构

下一格没有重新讲 GQA/因果 mask；你可以先看标记 `trace` 的几行。它只记录 dtype，不返回整张注意力矩阵，也不修改参数。

这个小模型用于**训练精度**实验，固定等长、无 padding；没有 KV cache、生成、变长请求或分布式训练接口。它也不加载前课检查点。不要把这个文件误当作包含所有历史功能的推理库。

In [12]:
class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.d = cfg, cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.d, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.d, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        inv = 10000.0 ** (-torch.arange(0, self.d, 2).float()/self.d)
        self.register_buffer("inv_freq", inv)

    def forward(self, x, trace=None):
        b, t, _ = x.shape
        q = self.q_proj(x).reshape(b, t, self.cfg.n_heads, self.d).transpose(1, 2)
        k = self.k_proj(x).reshape(b, t, self.cfg.n_kv_heads, self.d).transpose(1, 2)
        v = self.v_proj(x).reshape(b, t, self.cfg.n_kv_heads, self.d).transpose(1, 2)
        q, k = apply_rope(q, self.inv_freq), apply_rope(k, self.inv_freq)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0,
                                             enable_gqa=(self.cfg.n_heads != self.cfg.n_kv_heads))
        result = self.out_proj(out.transpose(1, 2).reshape(b, t, -1))
        if trace is not None:
            trace.update(q=str(q.dtype), k=str(k.dtype), v=str(v.dtype),
                         attention_core_output=str(out.dtype), attention_projection=str(result.dtype))
        return result


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg.d_model, cfg.eps), RMSNorm(cfg.d_model, cfg.eps)
        self.attn = Attention(cfg)
        self.gate = nn.Linear(cfg.d_model, cfg.ff_dim, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.ff_dim, bias=False)
        self.down = nn.Linear(cfg.ff_dim, cfg.d_model, bias=False)

    def forward(self, x, trace=None):
        normalized = self.norm1(x)
        x = x + self.attn(normalized, trace)
        h = self.norm2(x)
        gate = F.silu(self.gate(h))
        delta = self.down(gate * self.up(h))
        result = x + delta
        if trace is not None:
            trace.update(norm_output=str(normalized.dtype), residual=str(result.dtype),
                         gate_activation=str(gate.dtype), ffn_output=str(delta.dtype))
        return result


class TinyDecoder(nn.Module):
    def __init__(self, cfg=ModelConfig()):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.eps)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, ids, trace=None):
        if ids.dtype != torch.long or ids.ndim != 2 or min(ids.shape) <= 0:
            raise ValueError("ids 必须是非空 torch.long [B,T]")
        if ids.shape[1] > self.cfg.max_len:
            raise ValueError("超过本课最大序列长度")
        x = self.embedding(ids)
        if trace is not None:
            trace.update(input=str(ids.dtype), embedding=str(x.dtype))
        for block in self.blocks:
            x = block(x, trace)
        logits = self.head(self.norm(x))
        if trace is not None:
            trace["logits"] = str(logits.dtype)
        return logits


def new_model(cfg, seed=2401, device=torch.device("cpu")):
    # 初始化只在 CPU 消耗独立、暂存的 RNG 状态，再迁移；各精度分支用同一份 state。
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model = TinyDecoder(cfg)
    return model.to(device)


def token_loss(logits, targets):
    if targets.dtype != torch.long or targets.shape != logits.shape[:-1]:
        raise ValueError("targets 必须是与 logits 前两维匹配的 long 张量")
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))

## 7. 数据和能力探测

为了隔离数值实现，本课不再使用中文模板，而用一个可复现的符号任务：

```
[BOS, a, b, a, b, a, b, a, b, EOS]
```

普通符号编号为 4–15，`a != b`，一共 132 条不同序列。固定划分为 106 条训练与 26 条验证，标签只在这里右移一次。相同规则贯穿两侧，它不是自然语言理解或真实世界泛化基准。

所有精度分支用同一份初始化和同一个有放回 batch 采样日程。为了明确实际支持，我们不只读一个硬件标记，还运行整个模型的前向和反传。

CPU 的完整 FP16 训练按本课设计跳过，不是声称“CPU 完全不支持 FP16”。BF16 的可执行性也不是速度保证。

In [13]:
def make_data(cfg=ModelConfig(), seed=2402):
    # 0 留空，1 BOS，2 EOS，3 保留；其余是普通离散符号。
    # [BOS,a,b,a,b,a,b,a,b,EOS]；只在 Dataset 处移动一次标签。
    rows = [[1] + [a, b]*4 + [2]
            for a in range(4, cfg.vocab_size) for b in range(4, cfg.vocab_size) if a != b]
    full = torch.tensor(rows, dtype=torch.long)
    g = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(rows), generator=g)
    n_val = max(4, len(rows)//5)
    val_full, train_full = full[order[:n_val]], full[order[n_val:]]
    return {"train_x": train_full[:, :-1].contiguous(), "train_y": train_full[:, 1:].contiguous(),
            "val_x": val_full[:, :-1].contiguous(), "val_y": val_full[:, 1:].contiguous()}


def batch_schedule(n_samples, steps, batch_size, seed=2403):
    for v, n in ((n_samples, "n_samples"), (steps, "steps"), (batch_size, "batch_size")):
        positive_int(v, n)
    return torch.randint(n_samples, (steps, batch_size), generator=torch.Generator().manual_seed(seed))


def fingerprint(state):
    h = hashlib.sha256()
    for name, x in sorted(state.items()):
        h.update(name.encode()); h.update(str(x.dtype).encode()); h.update(str(tuple(x.shape)).encode())
        h.update(bytes(x.detach().cpu().contiguous().view(torch.uint8).reshape(-1).tolist()))
    return h.hexdigest()

In [14]:
def capability_probe(mode, device, cfg=ModelConfig()):
    validate_mode(mode)
    record = {"mode": mode, "device": str(device), "supported": False, "reason": None}
    if device.type == "cpu" and mode == "amp_fp16":
        record["reason"] = "本课 CPU 不运行完整 FP16 训练；这不是声称 CPU 绝对不支持 FP16。数值实验仍含 FP16。"
        return record
    if device.type == "cuda" and mode == "amp_bf16" and not torch.cuda.is_bf16_supported():
        record["reason"] = "当前 CUDA 环境未报告 BF16 支持"
        return record
    try:
        model = new_model(cfg, device=device)
        data = make_data(cfg)
        x, y = data["train_x"][:2].to(device), data["train_y"][:2].to(device)
        with amp_context(mode, device):
            logits = model(x)
            loss = token_loss(logits, y)
        loss.backward()
        if not bool(torch.isfinite(loss)) or not all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters()):
            raise FloatingPointError("探测的 loss/梯度不是有限值")
        wanted = MODE_DTYPES[mode]
        if logits.dtype != wanted:
            raise RuntimeError(f"预期 head 输出 {wanted}，实际 {logits.dtype}")
        record.update(supported=True, logits_dtype=str(logits.dtype), loss_dtype=str(loss.dtype))
    except (RuntimeError, TypeError, NotImplementedError, FloatingPointError) as exc:
        record["reason"] = f"{type(exc).__name__}: {exc}"
    return record


def precision_snapshot(cfg, state, x, y, mode, device):
    model = new_model(cfg, device=device)
    model.load_state_dict(state, strict=True)
    model.eval()
    trace = {}
    with amp_context(mode, device):
        logits = model(x.to(device), trace)
        loss = token_loss(logits, y.to(device))
    loss.backward()
    trace.update(loss=str(loss.dtype), parameters=sorted({str(p.dtype) for p in model.parameters()}),
                 leaf_gradients=sorted({str(p.grad.dtype) for p in model.parameters() if p.grad is not None}))
    return {"trace": trace, "loss": float(loss.detach()),
            "logits": logits.detach().float().cpu(),
            "gradient": torch.cat([p.grad.detach().float().reshape(-1).cpu() for p in model.parameters()]),
            "state_unchanged": fingerprint(model.state_dict()) == fingerprint(state)}

In [15]:
CFG = ModelConfig()
DATA = make_data(CFG)
print("训练/验证：", len(DATA["train_x"]), len(DATA["val_x"]))
print("第一条输入：", DATA["train_x"][0].tolist())
print("第一条目标：", DATA["train_y"][0].tolist())
print("模型参数量：", sum(p.numel() for p in new_model(CFG).parameters()))
PROBES = [capability_probe(m, DEVICE, CFG) for m in MODES]
for row in PROBES:
    print(row)

训练/验证： 106 26
第一条输入： [1, 5, 11, 5, 11, 5, 11, 5, 11]
第一条目标： [5, 11, 5, 11, 5, 11, 5, 11, 2]
模型参数量： 52464
{'mode': 'fp32', 'device': 'cpu', 'supported': True, 'reason': None, 'logits_dtype': 'torch.float32', 'loss_dtype': 'torch.float32'}
{'mode': 'amp_fp16', 'device': 'cpu', 'supported': False, 'reason': '本课 CPU 不运行完整 FP16 训练；这不是声称 CPU 绝对不支持 FP16。数值实验仍含 FP16。'}
{'mode': 'amp_bf16', 'device': 'cpu', 'supported': True, 'reason': None, 'logits_dtype': 'torch.bfloat16', 'loss_dtype': 'torch.float32'}


## 8. 固定权重，直接观察 dtype 和梯度误差

这一段不更新参数，只比较**同一组权重、同一批输入**的 FP32 与 AMP 前向/反向结果。

注意两种误差：输出的最大绝对误差，以及整组梯度相对于 FP32 参考的 L2 误差。BF16 的误差没有普遍适用的固定阈值；本课检查只针对这份小模型和输入。

在当前实现中，FP32 embedding 与低精度分支结果相加后，残差主路仍是 FP32。低精度 logits、FP32 CE loss、FP32 叶子梯度可以同时存在。

In [16]:
BASE_MODEL = new_model(CFG)
BASE_STATE = {k: v.detach().clone() for k, v in BASE_MODEL.state_dict().items()}
with fp32_reference_policy():
    reference = precision_snapshot(CFG, BASE_STATE, DATA["val_x"][:8], DATA["val_y"][:8], "fp32", DEVICE)
    for probe in PROBES:
        if not probe["supported"]:
            continue
        snap = precision_snapshot(CFG, BASE_STATE, DATA["val_x"][:8], DATA["val_y"][:8], probe["mode"], DEVICE)
        print("\n", probe["mode"])
        print(json.dumps(snap["trace"], ensure_ascii=False, indent=2))
        print("最大 logits 误差：", float((snap["logits"]-reference["logits"]).abs().max()))
        print("梯度相对 L2 误差：", float((snap["gradient"]-reference["gradient"]).norm()/reference["gradient"].norm()))
        print("参数未改动：", snap["state_unchanged"])


 fp32
{
  "input": "torch.int64",
  "embedding": "torch.float32",
  "q": "torch.float32",
  "k": "torch.float32",
  "v": "torch.float32",
  "attention_core_output": "torch.float32",
  "attention_projection": "torch.float32",
  "norm_output": "torch.float32",
  "residual": "torch.float32",
  "gate_activation": "torch.float32",
  "ffn_output": "torch.float32",
  "logits": "torch.float32",
  "loss": "torch.float32",
  "parameters": [
    "torch.float32"
  ],
  "leaf_gradients": [
    "torch.float32"
  ]
}
最大 logits 误差： 0.0
梯度相对 L2 误差： 0.0
参数未改动： True

 amp_bf16
{
  "input": "torch.int64",
  "embedding": "torch.float32",
  "q": "torch.bfloat16",
  "k": "torch.bfloat16",
  "v": "torch.bfloat16",
  "attention_core_output": "torch.bfloat16",
  "attention_projection": "torch.bfloat16",
  "norm_output": "torch.float32",
  "residual": "torch.float32",
  "gate_activation": "torch.bfloat16",
  "ffn_output": "torch.bfloat16",
  "logits": "torch.bfloat16",
  "loss": "torch.float32",
  "parameters":

## 9. 正确训练循环：看懂每个阶段再运行

本课完整实现使用启用/禁用的同一个 scaler 对象统一分支：FP16 启用，FP32/BF16 禁用。禁用时 `scale/step/update` 的相关接口退化为普通路径。

```python
optimizer.zero_grad(set_to_none=True)
with torch.autocast("cuda", dtype=torch.float16):
    logits = model(input_ids)
    loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
```

不要另外再调用 `optimizer.step()`，否则会重复更新。不要在同一次 step 里重复 unscale。前向 loss 非有限时，本课明确停止并提示排查；梯度溢出交由已启用的 scaler 跳步。

本课 `init_scale=1024` 是小模型实验设置，不是通用推荐值。默认样例不加入梯度累积；下一课再研究累积时缩放、裁剪与更新的边界。

In [17]:
class CountingAdamW(torch.optim.AdamW):
    """只统计实际 optimizer.step 调用；step 返回 None 不能用来判断 GradScaler 是否跳过。"""
    def __init__(self, params, **kwargs):
        super().__init__(params, **kwargs)
        self.actual_steps = 0

    def step(self, closure=None):
        result = super().step(closure=closure)
        self.actual_steps += 1
        return result


def make_scaler(mode, device):
    validate_mode(mode)
    # 1024 是本课小模型的演示初始 scale，不是对所有模型的推荐常数。
    return torch.amp.GradScaler(device.type, enabled=(mode == "amp_fp16"),
                               init_scale=1024.0, growth_interval=200)


def train_step(model, optimizer, scaler, x, y, mode, max_norm=1.0):
    if not math.isfinite(max_norm) or max_norm <= 0:
        raise ValueError("max_norm 必须正且有限")
    device = next(model.parameters()).device
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with amp_context(mode, device):
        logits = model(x)
        loss = token_loss(logits, y)
    if not bool(torch.isfinite(loss.detach())):
        raise FloatingPointError("前向 loss 已非有限；停止检查输入/计算，不能指望 loss scaling 修复。")
    before = optimizer.actual_steps
    old_scale = scaler.get_scale()
    scaler.scale(loss).backward()  # disabled 时等于普通 backward
    scaler.unscale_(optimizer)    # disabled 时无操作；enabled 时只执行一次
    norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm,
                                   error_if_nonfinite=not scaler.is_enabled())
    # GradScaler 已在 unscale_ 记录 inf/nan；发现时跳过 optimizer.step。
    scaler.step(optimizer)
    scaler.update()
    return {"loss": float(loss.detach()), "grad_norm_before_clip": scalar_text(float(norm)),
            "scale_before": old_scale, "scale_after": scaler.get_scale(),
            "optimizer_ran": optimizer.actual_steps > before}


@torch.no_grad()
def evaluate(model, x, y, mode="fp32"):
    was_training = model.training
    model.eval()
    try:
        with amp_context(mode, next(model.parameters()).device):
            logits = model(x)
            loss = token_loss(logits, y)
        return {"loss": float(loss), "accuracy": float((logits.argmax(-1) == y).float().mean())}
    finally:
        model.train(was_training)

## 10. 资源核算：参数、梯度与 Adam 状态还在占空间

“16 位比 32 位少一半”只针对**同样数量、对应 dtype 的元素载荷**。它不自动推广为整个训练程序的峰值显存减半。

本课的参数、叶子梯度、Adam 的一阶与二阶状态都保留 FP32。粗略按每个参数计算：

\[
4\text{（参数）}+4\text{（梯度）}+4\text{（一阶状态）}+4\text{（二阶状态）}=16\text{ bytes}
\]

这还没有算激活、临时低精度副本、算子工作区、buffer 和分配器开销；Adam 的 step 张量也额外占空间。实际核算下面会逐个数张量，而不是只套公式。

CUDA 报告另记录训练及周期验证期间的峰值 allocated。它不是系统显示的整进程 GPU 显存；CPU 时保存为 `null`，不填 0。

来源：[AMP 官方配方](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html)、[CUDA allocated 峰值](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html)。

In [18]:
def storage_inventory(model, optimizer):
    """只计现存 tensor 的元素载荷，不是峰值显存；统计时梯度尚未置 None。"""
    def total(tensors):
        items = list(tensors)
        return {"bytes": sum(t.numel()*t.element_size() for t in items),
                "dtypes": sorted({str(t.dtype) for t in items}), "tensors": len(items)}
    states = [v for state in optimizer.state.values() for v in state.values() if isinstance(v, torch.Tensor)]
    return {"parameters": total(model.parameters()),
            "gradients": total(p.grad for p in model.parameters() if p.grad is not None),
            "adam_tensors_including_steps": total(states),
            "buffers": total(model.buffers()),
            "note": "不包含中间激活、autocast 临时副本、算子工作区、缓存分配器及 Python 对象；不是显存峰值。"}


def train_mode(cfg, state, data, schedule, mode, device, lr=0.003):
    if not math.isfinite(lr) or lr <= 0:
        raise ValueError("lr 必须正且有限")
    model = new_model(cfg, device=device)
    model.load_state_dict(state, strict=True)
    initial_hash = fingerprint(model.state_dict())
    moved = {k: v.to(device) for k, v in data.items()}
    schedule_device = schedule.to(device)
    optimizer = CountingAdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scaler = make_scaler(mode, device)
    initial = evaluate(model, moved["val_x"], moved["val_y"])
    updates, history = [], []
    memory = None
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        baseline = torch.cuda.memory_allocated(device)
        torch.cuda.reset_peak_memory_stats(device)
    for i, indices in enumerate(schedule_device, 1):
        row = train_step(model, optimizer, scaler,
                         moved["train_x"][indices], moved["train_y"][indices], mode)
        row["attempt"] = i
        updates.append(row)
        if i == 1 or i % 20 == 0 or i == len(schedule):
            history.append({"attempt": i, "actual_updates": optimizer.actual_steps,
                            "validation_fp32": evaluate(model, moved["val_x"], moved["val_y"])})
    # 到这里的峰值包含初始化 optimizer state、训练、周期验证；不称为纯训练一步峰值。
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        peak = torch.cuda.max_memory_allocated(device)
        memory = {"baseline_allocated": baseline, "peak_allocated": peak,
                  "increment_over_baseline": peak - baseline,
                  "reserved_now": torch.cuda.memory_reserved(device),
                  "scope": "本次训练+周期验证，含 Adam 状态建立；输入/模型已在 baseline；非进程显存。"}
    final_fp32 = evaluate(model, moved["val_x"], moved["val_y"], "fp32")
    final_native = evaluate(model, moved["val_x"], moved["val_y"], mode)
    inventory = storage_inventory(model, optimizer)
    checkpoint = {"family": "lesson24_precision_v1", "config": asdict(cfg), "mode": mode,
                  "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                  "scaler_state": scaler.state_dict(),
                  "note": "无 optimizer/RNG/数据位置，不是精确断点续训检查点。"}
    report = {"mode": mode, "initial_fingerprint": initial_hash,
              "final_fingerprint": fingerprint(checkpoint["state_dict"]),
              "initial_validation_fp32": initial, "final_validation_fp32": final_fp32,
              "final_validation_native": final_native,
              "attempted_steps": len(schedule), "actual_optimizer_steps": optimizer.actual_steps,
              "skipped_steps": len(schedule) - optimizer.actual_steps,
              "inventory": inventory, "cuda_memory": memory,
              "history": history, "updates": updates}
    return report, checkpoint

## 11. 完整训练对照

每个可用模式从同一初始化开始，训练 80 次**尝试**，batch size 为 16，Dropout=0。公共参数与数据顺序不变。若某精度因溢出跳步，报告会显示实际更新次数不相同，而不会伪称三组完成了同样多的参数更新。

最终分别记录两种验证：

| 验证口径 | 回答的问题 |
|---|---|
| 所有最终权重统一用 FP32 前向 | 改变训练精度后，得到的参数怎样？ |
| 各自用本模式前向 | 训练后再使用该模式推理，会得到什么结果？ |

保存的是**最后状态**，不按验证集选择最佳轮次。本实验没有独立最终测试集。不要根据一次小数据实验宣布 BF16 等同 FP32、优于 FP32，或判断真实语言能力。

下一格定义完整流程（不会立即训练）。自动检查的定义在练习答案后面；这里先不调用检查，以免提前暴露答案。

In [19]:
def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def run_experiment(device=torch.device("cpu"), steps=80, batch_size=16,
                   requested="all", output_dir="lesson_24_outputs", perform_checks=True):
    positive_int(steps, "steps"); positive_int(batch_size, "batch_size")
    if requested not in ("all",)+MODES:
        raise ValueError("未知 requested 模式")
    cfg = ModelConfig()
    output_dir = Path(output_dir).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    with fp32_reference_policy() as policy:
        checks = run_checks(device) if perform_checks else None
        probes = [capability_probe(m, device, cfg) for m in MODES]
        selected = [p["mode"] for p in probes if p["supported"] and (requested == "all" or p["mode"] == requested)]
        if not selected:
            raise RuntimeError(f"请求的模式无法执行：{probes}")
        data = make_data(cfg)
        schedule = batch_schedule(len(data["train_x"]), steps, batch_size)
        base = new_model(cfg)
        state = {k: v.detach().clone() for k, v in base.state_dict().items()}
        reference = precision_snapshot(cfg, state, data["val_x"][:8], data["val_y"][:8], "fp32", device)
        numerical = {"format_table": format_table(), "representation": representation_demo(),
                     "gradient_underflow": underflow_demo(), "clipping_order": clipping_order_demo(),
                     "scaler_control": scaler_control_demo(device)}
        compare, runs, checkpoints = {}, {}, {}
        for mode in selected:
            snap = precision_snapshot(cfg, state, data["val_x"][:8], data["val_y"][:8], mode, device)
            compare[mode] = {"trace": snap["trace"], "loss": snap["loss"],
                             "state_unchanged": snap["state_unchanged"],
                             "max_logit_abs_error_vs_fp32": float((snap["logits"]-reference["logits"]).abs().max()),
                             "gradient_relative_l2_error_vs_fp32": float((snap["gradient"]-reference["gradient"]).norm()/reference["gradient"].norm().clamp_min(1e-12))}
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
            report, checkpoint = train_mode(cfg, state, data, schedule, mode, device)
            # 文件保存后重新加载，并在 CPU/FP32 验证，避免只比较内存中的同一对象。
            path = output_dir/f"{mode}.pt"
            torch.save(checkpoint, path)
            loaded = torch.load(path, map_location="cpu", weights_only=True)
            restored = new_model(ModelConfig(**loaded["config"]))
            restored.load_state_dict(loaded["state_dict"], strict=True)
            report["reload_fingerprint_matches"] = fingerprint(restored.state_dict()) == report["final_fingerprint"]
            if not report["reload_fingerprint_matches"]:
                raise AssertionError("保存重载指纹不一致")
            report["reload_cpu_validation"] = evaluate(restored, data["val_x"], data["val_y"])
            runs[mode] = report
            checkpoints[mode] = str(path.name)
            print(f"{mode}: FP32验证损失 {report['initial_validation_fp32']['loss']:.6f} -> "
                  f"{report['final_validation_fp32']['loss']:.6f}; "
                  f"实际更新 {report['actual_optimizer_steps']}/{steps}")
        report = {"environment": environment(device), "fp32_reference_policy": policy,
                  "model_config": asdict(cfg), "parameter_count": sum(p.numel() for p in base.parameters()),
                  "data": {"task": "[BOS,a,b,a,b,a,b,a,b,EOS] 的 next-token 预测；非自然语言",
                           "train_rows": len(data["train_x"]), "val_rows": len(data["val_x"]),
                           "input_length": 9, "train_x": data["train_x"].tolist(),
                           "val_x": data["val_x"].tolist(), "schedule": schedule.tolist()},
                  "checks": checks, "capabilities": probes, "numerics": numerical,
                  "fixed_weight_comparison": compare, "training": runs, "checkpoints": checkpoints,
                  "limits": ["CPU/GPU 不保证逐位一致；本课没有速度基准。",
                             "仅单种子合成任务；不用这些分数比较真实模型能力。",
                             "所有尝试使用同一 batch 顺序；若 FP16 跳步，实际更新次数不同，必须看报告。",
                             "验证统一 FP32 与模式原生推理分别报告；不选择最佳轮次，仅保存最后状态。",
                             "AMP 不改变存储中 FP32 权重 dtype；不等于半精度权重加载或整数低比特量化。",
                             "KV cache、生成、padding、分布式及精确续训不在本课小模型范围。"],
                  "sources": SOURCES}
        write_json(output_dir/"numerics.json", numerical)
        write_json(output_dir/"training.json", runs)
        write_json(output_dir/"experiment.json", report)
    print("输出目录：", output_dir)
    if checks: print("通过固定检查：", checks["passed"], "项")
    return report

In [20]:
REPORT = run_experiment(
    device=DEVICE,
    steps=STEPS,
    batch_size=BATCH_SIZE,
    requested=MODE,
    output_dir=OUTPUT_DIR,
    perform_checks=False,
)

fp32: FP32验证损失 2.921977 -> 0.769976; 实际更新 80/80


amp_bf16: FP32验证损失 2.921977 -> 0.771681; 实际更新 80/80
输出目录： /mnt/data/lesson24_validation/notebook_run/lesson_24_outputs


### 从报告里读结果

注意同一批次的 loss 本身也会经过不同数值路径。检查报告时应同时看 loss、实际更新次数、梯度范数、scaler 变化和 dtype。

固定种子只能控制这份程序的随机输入；不同 PyTorch/CPU/GPU/后端不保证逐位一致。本课没有计时基准，也没有“打开 AMP 必定加速”的断言。CPU BF16 跑通不是你的 3060 上的性能结果。

In [21]:
for name, run in REPORT["training"].items():
    print(name)
    print("  统一 FP32 验证：", run["final_validation_fp32"])
    print("  本模式验证：", run["final_validation_native"])
    print("  实际/尝试更新：", run["actual_optimizer_steps"], "/", run["attempted_steps"])
    print("  参数字节：", run["inventory"]["parameters"]["bytes"])
    print("  Adam张量字节：", run["inventory"]["adam_tensors_including_steps"]["bytes"])
    print("  CUDA内存：", run["cuda_memory"])
    print("  保存重载指纹一致：", run["reload_fingerprint_matches"])

fp32
  统一 FP32 验证： {'loss': 0.7699763774871826, 'accuracy': 0.7692307829856873}
  本模式验证： {'loss': 0.7699763774871826, 'accuracy': 0.7692307829856873}
  实际/尝试更新： 80 / 80
  参数字节： 209856
  Adam张量字节： 419796
  CUDA内存： None
  保存重载指纹一致： True
amp_bf16
  统一 FP32 验证： {'loss': 0.7716814279556274, 'accuracy': 0.7649572491645813}
  本模式验证： {'loss': 0.7720858454704285, 'accuracy': 0.7649572491645813}
  实际/尝试更新： 80 / 80
  参数字节： 209856
  Adam张量字节： 419796
  CUDA内存： None
  保存重载指纹一致： True


## 12. 在 RTX 3060 上怎么验证？

先保留这次 CPU 输出，再在项目目录运行对应脚本，例如：

```bash
python lesson_24_fp32_fp16_bf16_and_amp.py --device cuda --mode amp_fp16 --output-dir lesson_24_cuda_fp16
python lesson_24_fp32_fp16_bf16_and_amp.py --device cuda --mode amp_bf16 --output-dir lesson_24_cuda_bf16
```

你也可以用 `--device cuda --mode all` 一次跑全部可用分支。

报告既记录 `torch.cuda.is_bf16_supported` 的结果，也做实际完整模型前向/反向探测。较新接口可区分含模拟和原生支持；即使显示支持，也不等于某个内核一定更快。显式请求不可用模式时会报错。

请核对实际设备、CUDA 构建、模型 dtype、Q/K/V dtype、loss dtype、参数/梯度/Adam dtype、跳步次数、显存口径。遇到 OOM 先减 batch，而不是盲目把所有张量 `.half()`。

本课不安装驱动、CUDA Toolkit、编译扩展或 FlashAttention 包，也不需要修改旧课文件。

来源：[BF16 支持接口](https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_bf16_supported.html)。

## 13. 独立练习验收工具

下面只定义检查器，不给出答案。练习开关默认关闭，所以 **Run All 能通过不代表练习已完成**。完成后把开关改为 `True`，先看失败信息再对照后面的参考。

In [22]:
class Checks:
    def __init__(self):
        self.names = []

    def ok(self, condition, name):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def close(self, actual, expected, name, rtol=1e-5, atol=1e-7):
        torch.testing.assert_close(torch.as_tensor(actual), torch.as_tensor(expected), rtol=rtol, atol=atol)
        self.names.append(name)

    def raises(self, fn, error_type, name):
        try:
            fn()
        except error_type:
            self.names.append(name)
        else:
            raise AssertionError(f"{name}: 没有抛出 {error_type.__name__}")


def exercise_a_checks(fn):
    """fn(x): 用 FP32 计算最后一轴 RMS（带 eps），输出 FP32；不含权重。"""
    c = Checks()
    x = torch.tensor([[300.0, 400.0]], dtype=torch.float16)
    out = fn(x, 1e-6)
    c.ok(out.dtype == torch.float32, "A 输出 FP32")
    c.ok(bool(torch.isfinite(out).all()), "A 避免先平方溢出")
    c.close(out.square().mean(-1), [1.0], "A 正确末维尺度")
    c.close(fn(torch.zeros(2, 3), 1e-6), torch.zeros(2, 3), "A 零向量")
    xx = torch.tensor([[1., 2., 4.], [-3., 2., 1.]], requires_grad=True)
    got = fn(xx, 1e-6)
    expected = xx * torch.rsqrt(xx.square().mean(-1, keepdim=True) + 1e-6)
    c.close(got, expected, "A 保持每行独立")
    g1 = torch.autograd.grad(got.square().sum(), xx, retain_graph=True)[0]
    g2 = torch.autograd.grad(expected.square().sum(), xx)[0]
    c.close(g1, g2, "A 梯度一致", atol=2e-6)
    return c.names


def exercise_b_checks(fn):
    """fn(model,opt,scaler,x,y): 一次 FP32 SGD 的缩放/去缩放/裁剪更新。

    loss 统一为 (model(x)-y)^2.mean，max_norm=1；真实 CPU scaler 保证顺序可检查。
    """
    c = Checks()
    for scale in (1.0, 128.0):
        layer = nn.Linear(1, 1, bias=False)
        with torch.no_grad(): layer.weight.fill_(1.0)
        opt = torch.optim.SGD(layer.parameters(), lr=0.1)
        scaler = torch.amp.GradScaler("cpu", init_scale=scale, growth_interval=100)
        x, y = torch.ones(1, 1), torch.zeros(1, 1)
        # 故意放入旧梯度：检查学生是否清零。
        layer.weight.grad = torch.full_like(layer.weight, -100.0)
        loss = fn(layer, opt, scaler, x, y)
        c.close(layer.weight.detach(), [[0.9]], f"B scale={scale} 更新正确", atol=2e-6)
        c.ok(math.isfinite(float(loss)), f"B scale={scale} loss 有限")
        # 再跑一次，验证 update() 被调用，内部 scaler 状态已进入下一轮。
        fn(layer, opt, scaler, x, y)
        c.close(layer.weight.detach(), [[0.8]], f"B scale={scale} 可重复运行", atol=2e-6)
    return c.names


def exercise_c_checks(fn):
    """fn(model,x,mode) -> detached logits；保持模式、参数和原有梯度。"""
    c = Checks()
    model = nn.Linear(3, 2)
    x = torch.tensor([[1., 2., 3.]])
    original = copy.deepcopy(model.state_dict())
    for p in model.parameters(): p.grad = torch.ones_like(p)
    for was_training in (True, False):
        model.train(was_training)
        out = fn(model, x, "fp32")
        c.ok(model.training == was_training, f"C 恢复模式 {was_training}")
        c.ok(not out.requires_grad, f"C 关闭梯度 {was_training}")
        c.ok(out.shape == (1, 2), f"C 形状 {was_training}")
    c.ok(all(torch.equal(model.state_dict()[k], v) for k, v in original.items()), "C 参数不变")
    c.ok(all(torch.equal(p.grad, torch.ones_like(p)) for p in model.parameters()), "C 原有梯度不变")
    out = fn(model, x, "amp_bf16")
    c.ok(out.dtype == torch.bfloat16, "C autocast 确实生效")
    return c.names

### 练习 A · 实现安全的 RMS 归一化

实现 `stable_rms_ex(x, eps)`：沿最后一轴计算均方根，不减均值，不包含可学习权重；**先把输入变成 FP32 再平方**，返回 FP32。不要 `.detach()`，结果必须保留梯度。

用 `[300,400]` 的 FP16 输入验证不能溢出，再用零向量和多行输入验证 axis 与 eps。

In [23]:
def stable_rms_ex(x, eps=1e-6):
    raise NotImplementedError("先升 FP32，再计算最后一轴平方均值")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print("练习 A：", len(exercise_a_checks(stable_rms_ex)), "项通过")
else:
    print("练习 A 尚未启用；请自己实现。")

练习 A 尚未启用；请自己实现。


### 练习 B · 写对一次带缩放的训练更新

输入 `model, optimizer, scaler, x, y`，损失是均方误差 `(model(x)-y).square().mean()`。实现一次完整更新，裁剪阈值为 1，返回 detached loss。

这里故意用 CPU + FP32 模型 + 真实 scaler，**只检查控制顺序**，不测试 CUDA 内核。检查器会预先填入错误的旧梯度，并分别用 scale=1、128 执行两次更新。

需要做到：清零 → 前向 → scale/backward → unscale → clip → scaler.step → update。不要直接调用两次优化器。

In [24]:
def scaled_step_ex(model, optimizer, scaler, x, y):
    raise NotImplementedError("完整实现一次缩放更新，注意清零与裁剪顺序")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print("练习 B：", len(exercise_b_checks(scaled_step_ex)), "项通过")
else:
    print("练习 B 尚未启用；请自己实现。")

练习 B 尚未启用；请自己实现。


### 练习 C · 实现不改变模型状态的混合精度推理

实现 `predict_ex(model,x,mode)`，返回 detached logits。你需要使用 `eval()`、`no_grad()`、`amp_context`，并在结束（包括异常）后恢复原来 training 标志。

不要修改参数 dtype、权重或已有梯度。注意 `eval()` 不会自动关掉 autograd，也不会替你开启 autocast。

In [25]:
def predict_ex(model, x, mode):
    raise NotImplementedError("正确管理模式、梯度与 autocast")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    if capability_probe("amp_bf16", torch.device("cpu"))["supported"]:
        print("练习 C：", len(exercise_c_checks(predict_ex)), "项通过")
    else:
        print("当前 CPU 不支持这份 BF16 验收；先完成 FP32 部分。")
else:
    print("练习 C 尚未启用；请自己实现。")

练习 C 尚未启用；请自己实现。


---
## 14. 参考答案（先完成自己的版本）

这些答案使用独立函数名，不覆盖练习函数。运行答案只能证明答案通过，不代表你的实现完成。

A 的重点是先提升精度；B 的重点是 `unscale_` 在裁剪之前；C 的重点是用 `finally` 恢复原模式。

In [26]:
def answer_a(x, eps=1e-6):
    with torch.autocast(device_type=x.device.type, enabled=False):
        work = x.float()
        return work * torch.rsqrt(work.square().mean(-1, keepdim=True) + eps)


def answer_b(model, optimizer, scaler, x, y):
    optimizer.zero_grad(set_to_none=True)
    prediction = model(x)
    loss = (prediction-y).square().mean()
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    return loss.detach()


def answer_c(model, x, mode):
    old = model.training
    model.eval()
    try:
        with torch.no_grad(), amp_context(mode, x.device):
            return model(x).detach()
    finally:
        model.train(old)

In [27]:
print("参考 A：", len(exercise_a_checks(answer_a)), "项通过")
print("参考 B：", len(exercise_b_checks(answer_b)), "项通过")
if capability_probe("amp_bf16", torch.device("cpu"))["supported"]:
    print("参考 C：", len(exercise_c_checks(answer_c)), "项通过")

参考 A： 6 项通过
参考 B： 6 项通过
参考 C： 9 项通过


## 15. 自动检查与完整程序的一致性

自动检查覆盖格式、舍入、梯度下溢、缩放器跳步、裁剪顺序、模型因果性、前向/反向 dtype、真实参数更新、优化器存储与保存重载。

它还直接比较 `train_step` 的实际梯度与未缩放参考梯度，防止演示函数正确而完整训练函数顺序错误。

有条件的模式检查按实际探测决定数量；没有 CUDA 时不把 GPU 分支记作“通过”。以下检查不会训练完整的 80 步对照，只做必要的小步验证。

In [28]:
def run_checks(device=torch.device("cpu")):
    c = Checks()
    table = {r["name"]: r for r in format_table()}
    c.ok(table["FP16"]["max"] == 65504, "FP16 最大值")
    c.ok(table["FP16"]["eps_at_1"] == 2**-10, "FP16 eps")
    c.ok(table["BF16"]["eps_at_1"] == 2**-7, "BF16 eps")
    c.ok(table["FP32"]["eps_at_1"] == 2**-23, "FP32 eps")
    c.ok(table["BF16"]["smallest_normal"] == table["FP32"]["smallest_normal"], "BF16 与 FP32 最小正规值")
    c.ok(table["FP16"]["theoretical_smallest_subnormal"] == 2**-24, "tiny 不是最小 subnormal")
    c.ok(table["BF16"]["bytes"] == table["FP16"]["bytes"] == 2, "两个16位格式大小")
    r = representation_demo()
    c.ok(r["casts"]["fp16"][0] != r["casts"]["bf16"][0], "BF16 更粗的间隔")
    c.ok(r["casts"]["fp16"][1] == "+Infinity", "FP16 100000 溢出")
    c.ok(isinstance(r["casts"]["bf16"][1], float), "BF16 100000 有限")
    c.ok(r["casts"]["fp16"][2] == 0.0, "FP16 1e-8 舍入零")
    c.ok(r["casts"]["bf16"][2] > 0, "BF16 1e-8 非零")
    c.close(r["small_updates"]["fp32"], 0.99, "FP32 累积微小更新", atol=3e-6)
    c.ok(r["small_updates"]["fp16"] == 1.0, "FP16 参数小更新丢失")
    c.ok(r["small_updates"]["bf16"] == 1.0, "BF16 参数小更新丢失")
    c.ok(r["casting_back_does_not_recover"] == 1.0, "转回 FP32 不恢复信息")
    u = underflow_demo()
    c.ok(u["unscaled_path_gradient"] == 0, "不缩放的梯度消失")
    c.close(u["unscaled_after_backward"], 1e-8, "缩放后恢复近似梯度", rtol=0.003, atol=1e-12)
    c.ok(u["scaling_zero_after_backward"] == 0, "事后放大零无效")
    c.ok(u["scaling_overflowed_loss"] == "+Infinity", "缩放不是前向溢出修复")
    cl = clipping_order_demo()
    c.close(cl["correct_norm"], 1.0, "先还原再裁剪", atol=2e-6)
    c.close(cl["wrong_norm"], 1/128, "错误顺序产生错误尺度", atol=2e-6)
    sc = scaler_control_demo(device)
    c.ok(sc["actual_updates"] == 2, "3 次尝试只有2次更新")
    c.ok(sc["rows"][1]["before"] == sc["rows"][1]["after"], "inf 梯度跳过参数更新")
    c.ok(sc["rows"][1]["scale_after"] == 64.0, "inf 后 scale 减小")
    c.ok(all(row["loss_finite"] for row in sc["rows"]), "loss 有限不代表梯度有限")
    c.close(sc["rows"][-1]["after"], 0.64, "恢复有限梯度后继续更新")
    c.raises(lambda: ModelConfig(d_model=50), ValueError, "错误头维度被拒绝")
    c.raises(lambda: ModelConfig(n_kv_heads=3), ValueError, "非法 GQA 分组被拒绝")
    c.raises(lambda: amp_context("half", device), ValueError, "非法 precision 名字")
    cfg = ModelConfig()
    data = make_data(cfg)
    c.ok(data["train_x"].shape[0]+data["val_x"].shape[0] == 132, "合成样本数量")
    c.ok(torch.equal(data["train_x"][:, 1:], data["train_y"][:, :-1]), "标签只移位一次")
    c.ok(bool((data["train_x"][:, 0] == 1).all()), "BOS 输入")
    c.ok(bool((data["train_y"][:, -1] == 2).all()), "EOS 目标")
    train_rows = set(map(tuple, data["train_x"].tolist()))
    val_rows = set(map(tuple, data["val_x"].tolist()))
    c.ok(not train_rows.intersection(val_rows), "划分无相同序列")
    c.ok(torch.equal(batch_schedule(10, 4, 2), batch_schedule(10, 4, 2)), "固定 batch 顺序")
    model = new_model(cfg, device=device)
    x, y = data["train_x"][:3].to(device), data["train_y"][:3].to(device)
    logits = model(x)
    c.ok(logits.shape == (3, 9, 16), "模型输出形状")
    c.ok(all(p.dtype == torch.float32 for p in model.parameters()), "参数初始化 FP32")
    with torch.no_grad():
        changed = x.clone(); changed[:, -1] = 7
        c.close(model(changed)[:, :-1], logits.detach()[:, :-1], "完整模型因果性", atol=3e-6)
    out = evaluate(model, x, y)
    c.ok(model.training, "评估恢复 train 模式")
    c.ok(math.isfinite(out["loss"]), "FP32 loss 有限")
    norm = RMSNorm(2).to(device)
    xx = torch.tensor([[300., 400.]], device=device, dtype=torch.float16)
    c.ok(bool(torch.isfinite(norm(xx)).all()), "自定义 RMSNorm 先升精度")
    c.ok(norm(xx).dtype == torch.float16, "RMSNorm 输出按约定回到输入精度")
    expected_rms = xx.float() * torch.rsqrt(xx.float().square().mean(-1, keepdim=True) + 1e-6)
    c.close(norm(xx).float(), expected_rms, "真实 RMSNorm 高值数值正确", rtol=0.002, atol=0.002)
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    ref = precision_snapshot(cfg, state, x.cpu(), y.cpu(), "fp32", device)
    c.ok(ref["state_unchanged"], "梯度计算不修改参数")
    probes = [capability_probe(m, device, cfg) for m in MODES]
    for p in probes:
        if not p["supported"]:
            continue
        snap = precision_snapshot(cfg, state, x.cpu(), y.cpu(), p["mode"], device)
        c.ok(snap["trace"]["logits"] == str(MODE_DTYPES[p["mode"]]), f'{p["mode"]} logits dtype')
        c.ok(snap["trace"]["loss"] == "torch.float32", f'{p["mode"]} CE dtype')
        c.ok(snap["trace"]["parameters"] == ["torch.float32"], f'{p["mode"]} 参数保持 FP32')
        c.ok(snap["trace"]["leaf_gradients"] == ["torch.float32"], f'{p["mode"]} 叶子梯度 FP32')
        c.ok(bool(torch.isfinite(snap["gradient"]).all()), f'{p["mode"]} 梯度有限')
        c.ok((snap["logits"]-ref["logits"]).abs().max() < 0.08, f'{p["mode"]} 本小模型输出误差界')
        opt = CountingAdamW(model.parameters(), lr=0.001)
        before = fingerprint(model.state_dict())
        row = train_step(model, opt, make_scaler(p["mode"], device), x, y, p["mode"])
        c.ok(row["optimizer_ran"], f'{p["mode"]} 一次实际更新')
        c.ok(fingerprint(model.state_dict()) != before, f'{p["mode"]} 参数确实更新')
        inv = storage_inventory(model, opt)
        c.ok(inv["adam_tensors_including_steps"]["dtypes"] == ["torch.float32"], f'{p["mode"]} Adam 状态 FP32')
        c.ok(inv["parameters"]["bytes"] == sum(p.numel() for p in model.parameters())*4, "参数载荷核对")
    # 直接核对真实 train_step 的去缩放与裁剪顺序，不只测试演示函数。
    reference_model = new_model(cfg, device=device)
    actual_model = new_model(cfg, device=device)
    loss_ref = token_loss(reference_model(x), y)
    loss_ref.backward()
    nn.utils.clip_grad_norm_(reference_model.parameters(), 1.0)
    scaled_opt = CountingAdamW(actual_model.parameters(), lr=0.001)
    enabled_scaler = torch.amp.GradScaler(device.type, init_scale=128.0)
    train_step(actual_model, scaled_opt, enabled_scaler, x, y, "fp32")
    ref_grads = torch.cat([p.grad.reshape(-1) for p in reference_model.parameters()])
    real_grads = torch.cat([p.grad.reshape(-1) for p in actual_model.parameters()])
    c.close(real_grads, ref_grads, "真实训练步骤先 unscale 后 clip", rtol=2e-5, atol=2e-6)
    c.ok(real_grads.norm() <= 1.00001, "真实训练裁剪阈值")
    c.ok(scaled_opt.actual_steps == 1, "真实 scaler 仅更新一次")
    c.ok(all(p.dtype == torch.float32 for p in actual_model.parameters()), "实际缩放更新后参数仍 FP32")
    with tempfile.TemporaryDirectory() as folder:
        path = Path(folder)/"roundtrip.pt"
        torch.save({"config": asdict(cfg), "state_dict": model.state_dict()}, path)
        loaded = torch.load(path, map_location="cpu", weights_only=True)
        clone = new_model(ModelConfig(**loaded["config"]), device=device)
        clone.load_state_dict(loaded["state_dict"], strict=True)
        with torch.no_grad(): c.close(model(x), clone(x), "保存重载一致", atol=0, rtol=0)
    c.names.extend(exercise_a_checks(answer_a))
    c.names.extend(exercise_b_checks(answer_b))
    # CPU BF16 的练习验收按实际探测决定，不把平台缺失当作学生错误。
    if capability_probe("amp_bf16", torch.device("cpu"))["supported"]:
        c.names.extend(exercise_c_checks(answer_c))
    return {"passed": len(c.names), "checks": c.names, "device": str(device), "capabilities": probes}

In [29]:
with fp32_reference_policy():
    CHECK_REPORT = run_checks(DEVICE)
REPORT["checks"] = CHECK_REPORT
write_json(OUTPUT_DIR.resolve()/"experiment.json", REPORT)
print("固定检查通过：", CHECK_REPORT["passed"], "项")
print("报告：", (OUTPUT_DIR/"experiment.json").resolve())

固定检查通过： 91 项
报告： /mnt/data/lesson24_validation/notebook_run/lesson_24_outputs/experiment.json


## 16. 常见故障排查与本课验收

| 现象 | 优先检查 |
|---|---|
| CUDA 不可用 | 当前 Python/Jupyter 内核、PyTorch CUDA 构建、驱动；不假定 Notebook 与终端同环境 |
| 参数变成 FP16，更新报错 | 是否误用了 `model.half()`；本课 AMP 参数应保留 FP32 |
| loss 是 FP32 仍有 NaN/inf | 向前追踪中间结果；FP32 loss 不能修复此前的溢出 |
| scale 不断下降/频繁跳步 | 模型数值范围、输入、学习率、缩放前后梯度；不要只隐藏警告 |
| clipping 后更新异常小 | 是否在 `unscale_` 之前裁剪 |
| BF16 没变快或显存未减半 | 是否误把类型载荷推成系统性能；核对实际算子、形状和存储组成 |
| 训练不同模式结果不完全相等 | 先排除代码差异，再量化数值误差与实际更新数，不要求逐位一致 |

你应能独立回答：

1. 为什么 FP16 更容易溢出，而 BF16 在 1 附近更粗？
2. 为什么参数、梯度是 FP32，反向传播仍可能发生 FP16 下溢？
3. 为什么先 clip 再 unscale 改变了实际阈值？
4. 为什么一次循环不一定等于一次 optimizer 更新？
5. 为什么 AMP 不能直接推出整个训练程序的显存减半？

**下一课：第 25 课 · 梯度累积与激活检查点。** 把本课 AMP 接入受显存约束的训练，区分 micro-batch、有效 batch、参数更新步与激活重计算。

## 资料与运行说明

使用资料均为官方文档或原始论文；网上 `stable` 文档可能比本地 PyTorch 新，遇到 API 不一致先检查版本。

- [PyTorch 2.10 AMP](https://docs.pytorch.org/docs/2.10/amp.html)：autocast、算子规则、GradScaler。
- [AMP 官方示例](https://docs.pytorch.org/docs/stable/notes/amp_examples.html)：unscale、裁剪、跳步。
- [PyTorch Type Info](https://docs.pytorch.org/docs/stable/type_info.html)：eps、tiny、正规与次正规值。
- [NVIDIA 格式定义](https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/data-format-desc.html)：三种浮点格式。
- [Mixed Precision Training](https://arxiv.org/abs/1710.03740)：FP32 更新与 loss scaling 的背景。
- [数值精度与 TF32](https://docs.pytorch.org/docs/2.10/notes/numerical_accuracy.html)：误差、内部计算与设备差异。

对应脚本的默认命令：

```bash
python lesson_24_fp32_fp16_bf16_and_amp.py
python lesson_24_fp32_fp16_bf16_and_amp.py --checks-only
python lesson_24_fp32_fp16_bf16_and_amp.py --device cuda --mode all
```

生成 `experiment.json`、`numerics.json`、`training.json` 以及成功运行模式的 `.pt` 文件。模型文件含结构、FP32 权重与 scaler 状态供检查，但没有 optimizer、完整 RNG 和数据进度，**不是精确断点续训文件**。只加载自己创建或信任的文件。

本课没有生成整个课程压缩包，也没有改写项目 README。